## **Pergunta**: 

Por que um Incremental Loader não é possível nesta tabela, e por que o Full Loader é aceitável aqui
(volume, natureza dos dados).

## **Resposta**:

A tabela fornecedores_legado não possui coluna de criação ou última atualização que possbilita ter um rastreamento seguro de quais registros foram inseridos, atualizados ou deletados desde a última carga. O Full Loader é aceitável por se tratar de uma tabela de fornecedores e ter poucos dados quando comparado com as demais tabelas no banco de dados. Realizar o full loader tem o custo mas é aceitável neste cenário.

## Anotações

Conectar ao banco de dados

Consultar os dados via sql ( Carregar via pandas dataframe)

Gravar os dados em um arquivo (Json, Csv, parquet)
    Se salvar em parquet, talvez seria necessário atualizar os datatypes do dataframe para que seja compatível com o parquet

Para essa aula, a atividade necessita de parquet como output no bucket

```text
BRONZE/fornecedores_legado/
└── ano=2026/
    └── mes=11/
        └── dia=16/
            └── execucao=20261116T200000Z/
                ├── fornecedores_legado.parquet
                ├── _manifest.json
                └── _SUCCESS
```

## Case 1 Full Load Banco de Dados

### Configuração e conexão ao Banco de Dados

In [1]:
import os
from pathlib import Path
import pandas as pd
import psycopg2
from dotenv import load_dotenv
import io
import pyarrow.fs as fs
from datetime import datetime, timezone

# Usa o .env da raiz; no Docker, preserva as variáveis do container.
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / ".env").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Arquivo .env não encontrado na pasta do projeto.")

load_dotenv(PROJECT_ROOT / ".env", override=False)

def env(nome):
    valor = os.environ.get(nome, "").strip()
    if not valor:
        raise ValueError(f"Defina {nome} no .env ou no ambiente.")
    return valor


def conectar_postgres():
    return psycopg2.connect(
        host=env("POSTGRES_HOST"),
        port=int(env("POSTGRES_PORT")),
        dbname=env("POSTGRES_DB"),
        user=env("POSTGRES_USER"),
        password=env("POSTGRES_PASSWORD"),
        client_encoding="UTF8",
        application_name="02_batch_full_loader",
        connect_timeout=10,
        options="-c default_transaction_read_only=on",
    )


# Fecha a conexão anterior caso esta célula seja executada novamente.
if "conn" in globals() and not conn.closed:
    conn.close()

conn = conectar_postgres()
conn.autocommit = True

### Consultar query

In [ ]:
def consultar_informacoes(query: str, nome_tabela: str) -> pd.DataFrame:

    try:
        with conn.cursor() as cur:
            cur.execute(query)
            df = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description])

            return df
    except Exception:
        conn.close()
        raise


nome_tabela = "fornecedores_legado"

query = f"""     
    select
        id_fornecedor
        ,razao_social
        ,cnpj
        ,status_credito
        ,limite_credito
    from {nome_tabela}"""



df_fornecedor = consultar_informacoes(query, nome_tabela)
df_salvar_parquet = df_fornecedor.copy()

### Testes validações

#### Utilizar para teste - Quantidade de linhas a serem salvas no parquet


In [10]:
df_salvar_parquet = df_salvar_parquet.drop(index=0)

#### Utilizar para teste - Coluna inesperada


In [13]:
df_salvar_parquet = df_fornecedor.copy()
df_salvar_parquet["coluna_extra"] = "valor"

#### Utilizar para teste - Coluna ausente


In [22]:
df_salvar_parquet = df_fornecedor.copy()
df_salvar_parquet = df_salvar_parquet.drop(columns=["limite_credito"])

#### Utilizar para teste - DataFrame vazio


In [25]:
df_salvar_parquet = pd.DataFrame()

#### Utilizar para teste - Colunas duplicadas


In [ ]:
df_salvar_parquet = df_fornecedor.copy()
df_salvar_parquet = pd.concat(
    [df_salvar_parquet, df_salvar_parquet[["razao_social"]]],
    axis=1,
)
df_salvar_parquet.columns = [
    "id_fornecedor",
    "razao_social",
    "cnpj",
    "status_credito",
    "limite_credito",
    "razao_social",
]

#### Utilizar para teste - Valor textual inválido


In [ ]:
df_salvar_parquet = df_fornecedor.copy()
df_salvar_parquet["id_fornecedor"] = df_salvar_parquet["id_fornecedor"].astype(object)
df_salvar_parquet.loc[df_salvar_parquet.index[0], "id_fornecedor"] = 123

#### Utilizar para teste - Texto excedendo o tamanho permitido

In [ ]:
df_salvar_parquet = df_fornecedor.copy()
df_salvar_parquet.loc[df_salvar_parquet.index[0], "razao_social"] = "A" * 151

#### Utilizar para teste - Valor decimal com mais de duas casas

In [ ]:
df_salvar_parquet = df_fornecedor.copy()
df_salvar_parquet.loc[
    df_salvar_parquet.index[0],
    "limite_credito",
] = Decimal("1500.555")

#### Utilizar para teste - Valor float no limite de crédito

In [ ]:
df_salvar_parquet = df_fornecedor.copy()
df_salvar_parquet.loc[
    df_salvar_parquet.index[0],
    "limite_credito",
] = 1500.50

### Salvar os parquets no bucket

In [28]:
def validador_vazio(df: pd.DataFrame, nome_tabela: str) -> int:
    """
    Função para validar se o DataFrame está vazio. Se estiver, retorna 1 e encerra a execução.
    """

    if df.empty:
        return 1
    else:
        return 0


def validador_colunas(df: pd.DataFrame, nome_tabela: str) -> int:
    """
    Função para validar se o DataFrame possui colunas. Se não possuir, retorna 1 e encerra a execução.
    """

    if df.shape[1] == 0:
        return 1
    else:
        return 0


def validador_tipagem(df: pd.DataFrame, nome_tabela: str) -> int:
    """Valida o schema e os valores; retorna 0 no sucesso e 1 na falha."""
    from decimal import Decimal, InvalidOperation, localcontext

    schema_texto = {
        "id_fornecedor": 50,
        "razao_social": 150,
        "cnpj": 14,
        "status_credito": 20,
    }
    colunas_esperadas = set(schema_texto) | {"limite_credito"}
    erros = []
    faltantes = colunas_esperadas - set(df.columns)
    extras = set(df.columns) - colunas_esperadas

    if faltantes:
        erros.append(f"Colunas ausentes: {sorted(faltantes)}")
    if extras:
        erros.append(f"Colunas inesperadas: {sorted(extras)}")
    if df.columns.has_duplicates:
        print(f"Falha na validação de {nome_tabela}: colunas duplicadas.")
        return 1

    for coluna, tamanho in schema_texto.items():
        if coluna not in df.columns:
            continue
        
        # O schema informado permite NULL.
        valores = df[coluna].dropna()
        invalidos = valores.map(
            lambda valor: not isinstance(valor, str) or len(valor) > tamanho
        ).astype(bool)
        if invalidos.any():
            erros.append(
                f"{coluna}: {invalidos.sum()} valores fora de VARCHAR({tamanho}). "
                f"Índices de exemplo: {valores.index[invalidos].tolist()[:5]}"
            )

    def numeric_valido(valor):
        # psycopg2 retorna NUMERIC como Decimal; floats não garantem precisão.
        if isinstance(valor, bool) or not isinstance(valor, (Decimal, int)):
            return False
        numero = Decimal(valor)
        if not numero.is_finite() or abs(numero) >= Decimal("10000000000"):
            return False
        try:
            with localcontext() as contexto:
                contexto.prec = 28
                return numero == numero.quantize(Decimal("0.01"))
        except InvalidOperation:
            return False

    if "limite_credito" in df.columns:
        valores = df["limite_credito"].dropna()
        invalidos = ~valores.map(numeric_valido).astype(bool)
        if invalidos.any():
            erros.append(
                f"limite_credito: {invalidos.sum()} valores fora de NUMERIC(12,2). "
                f"Índices de exemplo: {valores.index[invalidos].tolist()[:5]}"
            )

    if erros:
        print(f"Falha na validação de {nome_tabela}:")
        for erro in erros:
            print(f"- {erro}")
        return 1

    print(f"Schema de {nome_tabela} validado com sucesso.")
    return 0


def validador_quantidade(df: pd.DataFrame, nome_tabela: str) -> int:
    """
    Função para validar a quantidade de registros no DataFrame em comparação com a tabela de destino. Se a quantidade fo diferente retorna 1 e encerra a execução.
    """

    query = f"SELECT COUNT(*) FROM {nome_tabela}"
    with conn.cursor() as cur:
        cur.execute(query)
        quantidade_tabela = cur.fetchone()[0]

    quantidade_df = len(df)

    if quantidade_df != quantidade_tabela:
        print(
            f"Quantidade de registros no DataFrame ({quantidade_df}) é diferente da tabela de destino ({quantidade_tabela})."
        )
        return 1
    else:
        return 0


def validador_aderencia(df: pd.DataFrame, nome_tabela: str) -> int:
    resultados = {
        "vazio": validador_vazio(df, nome_tabela),
        "colunas": validador_colunas(df, nome_tabela),
        "tipagem": validador_tipagem(df, nome_tabela),
        "quantidade": validador_quantidade(df, nome_tabela),
    }

    for nome, resultado in resultados.items():
        status = "SUCESSO" if resultado == 0 else "FALHA"
        print(f"{nome}: {status}")

    return int(any(resultado != 0 for resultado in resultados.values()))
    

def salvar_parquet_s3(df: pd.DataFrame, nome_tabela: str) -> None:
    """
    Salva um DataFrame em formato Parquet no MinIO/S3.
    """

    buffer = io.BytesIO()
    df.to_parquet(buffer, engine="pyarrow", index=False)

    minio = fs.S3FileSystem(
        access_key=env("MINIO_ROOT_USER"),
        secret_key=env("MINIO_ROOT_PASSWORD"),
        endpoint_override=env("S3_ENDPOINT_URL"),
        region="us-east-1",
    )

    timestamp = datetime.now(timezone.utc)
    ano = timestamp.strftime("%Y")
    mes = timestamp.strftime("%m")
    dia = timestamp.strftime("%d")
    execucao = timestamp.strftime("%Y%m%dT%H%M%SZ")

    resultado_aderencia = validador_aderencia(df, nome_tabela)

    if resultado_aderencia != 0:
        raise ValueError(
            f"Falha na aderência de {nome_tabela}. Gravação cancelada."
        )
    else:
        print(f"Aderência de {nome_tabela} validada com sucesso. Gravando arquivo...")

        caminho_diretorio = (
            f"{env('MINIO_BUCKET_BRONZE')}/{nome_tabela}/"
            f"ano={ano}/mes={mes}/dia={dia}/execucao={execucao}"
        )

        caminho_parquet = f"{caminho_diretorio}/{nome_tabela}.parquet"
        caminho_success = f"{caminho_diretorio}/_SUCCESS"

        with minio.open_output_stream(caminho_parquet) as destino:
            destino.write(buffer.getvalue())

        # Cria o objeto _SUCCESS com 0 bytes
        with minio.open_output_stream(caminho_success):
            pass

        print(f"Parquet salvo em {caminho_parquet}")
        print(f"Marcador criado em {caminho_success}")


salvar_parquet_s3(df_salvar_parquet, nome_tabela)

Schema de fornecedores_legado validado com sucesso.
vazio: SUCESSO
colunas: SUCESSO
tipagem: SUCESSO
quantidade: SUCESSO
Aderência de fornecedores_legado validada com sucesso. Gravando arquivo...
Parquet salvo em bronze/fornecedores_legado/ano=2026/mes=11/dia=23/execucao=20261123T231815Z/fornecedores_legado.parquet
Marcador criado em bronze/fornecedores_legado/ano=2026/mes=11/dia=23/execucao=20261123T231815Z/_SUCCESS


In [98]:
def ler_parquet_minio(caminho_parquet: str) -> pd.DataFrame:
    minio = fs.S3FileSystem(
        access_key=env("MINIO_ROOT_USER"),
        secret_key=env("MINIO_ROOT_PASSWORD"),
        endpoint_override=env("S3_ENDPOINT_URL"),
        region="us-east-1",
    )

    with minio.open_input_file(caminho_parquet) as arquivo:
        return pd.read_parquet(arquivo, engine="pyarrow")

caminho = "bronze/fornecedores_legado/ano=2026/mes=11/dia=17/execucao=20261117T031924Z/fornecedores_legado.parquet"

df_parquet_minio = ler_parquet_minio(caminho)
print(f"Shape: {df_parquet_minio.shape}")
print(f"Dtypes:\n{df_parquet_minio.dtypes}")

Shape: (50, 5)
Dtypes:
id_fornecedor        str
razao_social         str
cnpj                 str
status_credito       str
limite_credito    object
dtype: object


In [99]:
df_parquet_minio.head()

,id_fornecedor,razao_social,cnpj,status_credito,limite_credito
0,FORN-BDD640FB06,Alves e Filhos Ltda.,39250467000125,EM_ANALISE,174344.14
1,FORN-1A3D1FA7BC,Andrade S/A,13846092000177,BLOQUEADO,375989.28
2,FORN-0822E8F36C,Câmara - ME,94061325000128,APROVADO,72877.81
3,FORN-9A1DE64481,Cunha Teixeira Ltda. S/A,16928450000189,APROVADO,386645.18
4,FORN-B38A088CA6,Nascimento S/A,61495723000101,BLOQUEADO,291534.98
